In [1]:
"""
To run this test, download the OPSD dataset folder from 
https://data.open-power-system-data.org/time_series/
and place the 'opsd-time_series-2020-10-06' folder as a top-level subfolder in 
datasets_and_dataloaders. Then run this script.
"""

import os
import sys
import numpy as np
import pandas as pd
import pyscamp

import random
import importlib

current_dir = os.getcwd()
sys.path.append(current_dir)
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)

from datasets_and_dataloaders.dataloader import load_opsd_data


import mplot_python.MINT as M
importlib.reload(M)

from mplot_python.MINT import processAll, nnrobustpca_stable_pcp


In [2]:
data_path = os.path.join(parent_dir, 'datasets_and_dataloaders', 'opsd-time_series-2020-10-06')
np.random.seed(158)

df = load_opsd_data(data_path=data_path)
print(df.head(30))
print(df.tail())

for column in df.columns:
    if column == "utc_timestamp":
        continue

    series = df[column].copy()
    mask = series.isna()

    # --- Identify NaN runs ---
    group = (mask != mask.shift()).cumsum()
    nan_groups = series[mask].groupby(group[mask])

    # Compute column-wide mean and std for random filling
    mean = np.nanmean(series)
    sigma = np.nanstd(series)
    rand_background = np.random.normal(mean, sigma, len(series))

    for g, idxs in nan_groups.groups.items():
        run_length = len(idxs)
        start = idxs[0]
        end = idxs[-1]

        print(f"Imputing {run_length} NaNs in column '{column}' at positions {start}-{end} with random Gaussian fill.")

        series.iloc[idxs] = rand_background[idxs]

    df[column] = series

Looking for files in: c:\Users\masia\mint2\datasets_and_dataloaders\opsd-time_series-2020-10-06
['datapackage.json', 'README.md', 'time_series.xlsx', 'time_series_15min_singleindex.csv', 'time_series_30min_singleindex.csv', 'time_series_60min_singleindex.csv']
Loaded OPSD dataset with shape: (50401, 300)
           utc_timestamp  Austria  Cyprus  Germany  Denmark  Estonia    Spain  \
0   2014-12-31T23:00:00Z      NaN     NaN      NaN      NaN      NaN      NaN   
1   2015-01-01T00:00:00Z   5946.0     NaN  41151.0      NaN      NaN      NaN   
2   2015-01-01T01:00:00Z   5726.0     NaN  40135.0  3100.02    764.7  22734.0   
3   2015-01-01T02:00:00Z   5347.0     NaN  39106.0  2980.39    749.8  21286.0   
4   2015-01-01T03:00:00Z   5249.0     NaN  38765.0  2933.49    746.5  20264.0   
5   2015-01-01T04:00:00Z   5309.0     NaN  38941.0  2941.54    754.6  19905.0   
6   2015-01-01T05:00:00Z   5574.0     NaN  39045.0  2999.89    771.2  20010.0   
7   2015-01-01T06:00:00Z   5925.0     NaN  402

In [3]:
len(df) / 167

301.8023952095808

In [4]:
import numpy as np

np.random.seed(158)

subsequenceLength = 167
col_list = [c for c in df.columns if c != "utc_timestamp"]
exclude = []

print("filtering sensors...")
for idx, sensor_name in enumerate(col_list):

    print("Processing sensor:", sensor_name)
    series = df[sensor_name].to_numpy().astype(np.float32)    

    mplot = pyscamp.abjoin_matrix(
        series, series, subsequenceLength,
        mheight=418, mwidth=418, threshold=-1
    )

    if np.isnan(mplot).any():
        print(sensor_name + " excluded due to NaNs in MATRIX PROFILE")
        exclude.append(idx)
        
col_list = [col for i, col in enumerate(col_list) if i not in exclude]



results =  processAll(col_list, df, subsequenceLength, Mheight = 300, Mwidth = 300, name = "OPSD")

filtering sensors...
Processing sensor: Austria
Processing sensor: Cyprus
Processing sensor: Germany
Processing sensor: Denmark
Processing sensor: Estonia
Processing sensor: Spain
Processing sensor: Great Britain
Processing sensor: United Kingdom
Processing sensor: Greece
Processing sensor: Croatia
Processing sensor: Hungary
Processing sensor: Italy
Processing sensor: Lithuania
Processing sensor: Latvia
Processing sensor: Norway
Processing sensor: Portugal
Processing sensor: Sweden
Processing sensor: Slovakia
===TENSOR CALCULATION===
calculating mplots...
0 Austria
beginning RPCA...
RPCA done! Converged in 87 iteration(s).
tensor(1112.1064, dtype=torch.float64)
tensor(73.7753, dtype=torch.float64)
1 Cyprus
beginning RPCA...
RPCA done! Converged in 23 iteration(s).
tensor(3804.4653, dtype=torch.float64)
tensor(165.3606, dtype=torch.float64)
2 Germany
beginning RPCA...
RPCA done! Converged in 116 iteration(s).
tensor(822.5064, dtype=torch.float64)
tensor(66.2224, dtype=torch.float64)
3 D

In [5]:
from mplot_python.MINT import plotMatrixRaw
import numpy as np

A,B,C = results.low_rank_factors

A = np.abs(A)
B = np.abs(B)
C = np.abs(C)

color_map = "viridis"

plotMatrixRaw(A, "OPSD_A", "OPSDj", colormap = color_map)
plotMatrixRaw(B, "OPSD_B", "OPSDj", colormap = color_map)
plotMatrixRaw(C, "OPSD_C", "OPSDj", colormap = color_map)

In [6]:
pd.to_datetime(df.loc[0, "utc_timestamp"]) + pd.Timedelta(days = 7*90)

Timestamp('2016-09-21 23:00:00+0000', tz='UTC')

In [7]:
component_indices = range(C.shape[1])
s = 5
for i in component_indices:
    print(f"Component {i}")
    component = C[:, i]
    component_abs = np.abs(component)

    # Top-s indices in descending order
    ind = np.argpartition(component_abs, -s)[-s:]
    top_s_indices = ind[np.argsort(component_abs[ind])]
    top_s_indices = np.flip(top_s_indices)
    
    for j in top_s_indices:
        print(j, pd.to_datetime(df.loc[0, "utc_timestamp"]) + pd.Timedelta(days = 7*j), C[j,i])

Component 0
147 2017-10-25 23:00:00+00:00 0.08180396
251 2019-10-23 23:00:00+00:00 0.08012189
253 2019-11-06 23:00:00+00:00 0.07984084
145 2017-10-11 23:00:00+00:00 0.07972849
201 2018-11-07 23:00:00+00:00 0.07968171
Component 1
51 2015-12-23 23:00:00+00:00 0.06927575
260 2019-12-25 23:00:00+00:00 0.06722258
147 2017-10-25 23:00:00+00:00 0.067200616
101 2016-12-07 23:00:00+00:00 0.06705663
0 2014-12-31 23:00:00+00:00 0.06606313
Component 2
150 2017-11-15 23:00:00+00:00 0.09601117
214 2019-02-06 23:00:00+00:00 0.09568009
106 2017-01-11 23:00:00+00:00 0.09565623
160 2018-01-24 23:00:00+00:00 0.095020026
44 2015-11-04 23:00:00+00:00 0.094782725
Component 3
192 2018-09-05 23:00:00+00:00 0.073521525
191 2018-08-29 23:00:00+00:00 0.07331367
139 2017-08-30 23:00:00+00:00 0.073222876
89 2016-09-14 23:00:00+00:00 0.072347365
88 2016-09-07 23:00:00+00:00 0.07117062


In [8]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd

start = pd.to_datetime(df.loc[0, "utc_timestamp"])
x = start + pd.to_timedelta([7 * i for i in range(C.shape[0])], unit="D")

fig = make_subplots(
    rows=len(component_indices),
    cols=1,
    shared_xaxes=True,
    subplot_titles=[f"Component {i + 1}" for i in component_indices],
    vertical_spacing=0.10,
)

for row, i in enumerate(component_indices, start=1):
    fig.add_trace(
        go.Scatter(
            x=x,
            y=C[:, i],
            mode="lines",
            showlegend=False,
        ),
        row=row,
        col=1,
    )

# Show date labels on every subplot
fig.update_xaxes(showticklabels=True)


fig.update_layout(
    height=250 * len(component_indices),
    template="plotly_white",
)

fig.update_layout(margin=dict(l=90))

fig.show()

In [9]:
print(pd.to_datetime(df.loc[0, "utc_timestamp"]) + pd.Timedelta(days = 7*88))
print(pd.to_datetime(df.loc[0, "utc_timestamp"]) + pd.Timedelta(days = 7*98))
print(pd.to_datetime(df.loc[0, "utc_timestamp"]) + pd.Timedelta(days = 7*103))
print(pd.to_datetime(df.loc[0, "utc_timestamp"]) + pd.Timedelta(days = 7*106))
print(pd.to_datetime(df.loc[0, "utc_timestamp"]) + pd.Timedelta(days = 7*106))
print(pd.to_datetime(df.loc[0, "utc_timestamp"]) + pd.Timedelta(days = 7*117))

2016-09-07 23:00:00+00:00
2016-11-16 23:00:00+00:00
2016-12-21 23:00:00+00:00
2017-01-11 23:00:00+00:00
2017-01-11 23:00:00+00:00
2017-03-29 23:00:00+00:00
